In [5]:
import tensorflow as tf
import os, datetime, random
import tensorflow_recommenders as tfrs
from google.cloud import bigquery
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
import pprint
from typing import Dict, Text
import logging
import faiss
from keras_tuner import HyperModel
from keras_tuner.tuners import BayesianOptimization, Hyperband
from keras_tuner import Objective
import optuna

import xgboost as xgb
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

2025-01-26 18:57:14.729009: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/dmmckinn/repos/movie_recommender/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Check for GPU availability and set memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logger.info(f"GPUs available: {len(gpus)}")
    except RuntimeError as e:
        logger.warning(e)
else:
    logger.info("No GPUs available.")

2025-01-26 18:57:24,821 - INFO - No GPUs available.


In [7]:
# Define the BigQuery table and project details
PROJECT_ID = 'oolola'
DATASET_ID = 'movie_data'
TABLE_ID   = 'preprocessed_data'
timestamp  = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
output_dir = 'gs://movie-data-1/trained-model'
# Function to load movies from BigQuery
def load_movies_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT title, genres
        FROM `{PROJECT_ID}.{DATASET_ID}.preprocessed_movies`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading movies from BigQuery: {e}")
        raise
# Function to load ratings from BigQuery
def load_ratings_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT user_id, title, rating
        FROM `{PROJECT_ID}.{DATASET_ID}.ratings_with_titles`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading ratings from BigQuery: {e}")
        raise

In [8]:
movies_bq = load_movies_bq()
movies_dict = {key: list(value) for key, value in movies_bq[['title', 'genres']].to_dict(orient='list').items()}

In [9]:
ratings_bq = load_ratings_bq()
ratings_dict = {key: list(value) for key, value in ratings_bq[['title', 'user_id', 'rating']].to_dict(orient='list').items()}

In [10]:
ratings = tf.data.Dataset.from_tensor_slices(ratings_dict)

2025-01-26 18:57:41.467167: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [11]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'rating': 2.0, 'title': b'Initiation', 'user_id': 56703}


In [12]:
movies = tf.data.Dataset.from_tensor_slices(movies_dict)

In [13]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]),
 'title': b'Siberian Sniper'}


In [14]:
ratings = ratings.map(lambda x: {
            "title": x["title"],
            "user_id": x["user_id"],
            "rating": x["rating"]
        })
movies = movies.map(lambda x: {
    "title": x["title"],
    "genres": x["genres"]
})

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


2025-01-26 18:57:42,749 - WARNING - From /Users/dmmckinn/repos/movie_recommender/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/pyct/static_analysis/liveness.py:83: Analyzer.lamba_check (from tensorflow.python.autograph.pyct.static_analysis.liveness) is deprecated and will be removed after 2023-09-23.
Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


In [15]:
# print unique titles lenght from ratings and movies
print(f"Unique titles in ratings: {len(np.unique(ratings_dict['title']))}")
print(f"Unique titles in movies: {len(np.unique(movies_dict['title']))}")

Unique titles in ratings: 5283
Unique titles in movies: 5303


In [16]:
# Create a dictionary from the movies dataset
movies_dict = {movie["title"].numpy().decode('utf-8'): movie["genres"].numpy() for movie in movies}

# Combine the ratings and movies datasets only for the movies that are in the ratings dataset
def combine_datasets(rating):
    def lookup_genres(title):
        title_str = title.numpy().decode('utf-8')  # Convert to numpy and decode the title from bytes to string
        return movies_dict.get(title_str, [0] * 19)

    genres = tf.py_function(
        func=lookup_genres,
        inp=[rating["title"]],
        Tout=tf.int32
    )
    genres.set_shape([19])
    rating["genres"] = genres
    return rating

combined_dataset = ratings.map(combine_datasets)

In [17]:
# print one example of combined dataset
for x in combined_dataset.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
      dtype=int32),
 'rating': 2.0,
 'title': b'Initiation',
 'user_id': 56703}


In [18]:
combined_dataset = combined_dataset.map(lambda x: {
    "title": x["title"],
    "user_id": x["user_id"],
    "genres": x["genres"],
    "rating": x["rating"]
}, num_parallel_calls=tf.data.AUTOTUNE)

In [23]:
# Set the seed for reproducibility
tf.random.set_seed(42)

# Shuffle the dataset with a large buffer size
# Ensure the buffer size is large enough to cover randomness but not too large to exhaust memory
shuffle_buffer_size = 200000  # Smaller buffer size for faster shuffling
shuffled = combined_dataset.shuffle(buffer_size=shuffle_buffer_size, seed=42, reshuffle_each_iteration=True)

# Calculate relative proportions for splitting
train_ratio = 0.8

ds_length = int(tf.data.experimental.cardinality(shuffled).numpy())
print(f"Length of the dataset: {ds_length}")

# Define the split function for large datasets
def split_dataset(dataset, train_ratio):
    # Determine split points
    trainds = dataset.take(int(train_ratio * ds_length))
    testds = dataset.skip(int(train_ratio * ds_length))

    return trainds, testds

# Perform the split
trainds, testds = split_dataset(shuffled, train_ratio)

# Optimize performance with prefetching
train_batch_size =  128
test_batch_size = 64

# Optimize datasets with batching, caching, and prefetching
trainds = trainds.batch(train_batch_size).cache().prefetch(tf.data.AUTOTUNE)
testds = testds.batch(test_batch_size).cache().prefetch(tf.data.AUTOTUNE)

Length of the dataset: 174490


In [26]:
titles = movies.batch(100000).map(lambda x: x["title"])
user_ids = ratings.batch(1000000).map(lambda x: x["user_id"])
genres = movies.batch(100000).map(lambda x: {
        "title": x["title"],
        "genres": x["genres"]
    })

In [27]:
unique_titles = np.unique(np.concatenate(list(titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))
unique_genres = [
            'Action',
            'Adventure',
            'Animation',
            'Children',
            'Comedy',
            'Crime',
            'Drama',
            'Documentary',
            'Fantasy',
            'Film-Noir',
            'Horror',
            'IMAX',
            'Musical',
            'Mystery',
            'Romance',
            'Sci-Fi',
            'Thriller',
            'War',
            'Western'
        ]

In [28]:
print(f"Unique titles: {len(unique_titles)}")

Unique titles: 5303


In [29]:
class RecommendationModel(tfrs.Model):
    def __init__(self, user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight=1.0, retrieval_weight=1.0):
        super().__init__()
        self.user_model = user_model
        self.movie_model = movie_model
        self.genre_model = genre_model
        self.rating_model = rating_model
        self.rating_task = rating_task
        self.retrieval_task = retrieval_task
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def call(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        movie_embeddings = self.movie_model(features["title"])
        genre_embeddings = self.genre_model(features["genres"])
        rating_predictions = self.rating_model([features["user_id"], features["title"], features["genres"]])
        return user_embeddings, movie_embeddings, genre_embeddings, rating_predictions

    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        ratings = features.pop("rating")
        user_embeddings, movie_embeddings, genre_embeddings, rating_predictions = self(features)
        rating_loss = self.rating_task(labels=ratings, predictions=rating_predictions)
        retrieval_loss = self.retrieval_task(user_embeddings, movie_embeddings)
        return (self.rating_weight * rating_loss
                + self.retrieval_weight * retrieval_loss)

In [18]:
# # Function to create the model
# def create_model(unique_user_ids, unique_movie_ids, num_genres, rating_weight=1.0, retrieval_weight=1.0):
#     embedding_dimension = 128
#     user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
#     movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
#     genre_input = tf.keras.layers.Input(shape=(num_genres,), dtype=tf.float32, name='genres')

#     user_lookup = tf.keras.layers.IntegerLookup(vocabulary=unique_user_ids, mask_token=None)
#     movie_lookup = tf.keras.layers.StringLookup(vocabulary=unique_titles, mask_token=None)

#     user_embedding = tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
#     movie_embedding = tf.keras.layers.Embedding(len(unique_movie_ids) + 1, embedding_dimension)(movie_lookup(movie_input))
#     genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

#     concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

#     dense_1 = tf.keras.layers.Dense(256, activation="relu")(concatenated_embeddings)
#     dropout_1 = tf.keras.layers.Dropout(0.5)(dense_1)
#     dense_2 = tf.keras.layers.Dense(128, activation="relu")(dropout_1)
#     dropout_2 = tf.keras.layers.Dropout(0.5)(dense_2)
#     rating_output = tf.keras.layers.Dense(1)(dropout_2)


#     user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
#     movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
#     genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
#     rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

#     metrics = tfrs.metrics.FactorizedTopK(
#         candidates=tf.data.Dataset.from_tensor_slices(unique_movie_ids).batch(128).map(movie_model)
#     )
#     rating_task = tfrs.tasks.Ranking(
#         loss=tf.keras.losses.MeanSquaredError(),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()],
#     )
#     retrieval_task = tfrs.tasks.Retrieval(
#         metrics=metrics
#     )

#     model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight, retrieval_weight)
#     lr_schedule_ed = tf.keras.optimizers.schedules.ExponentialDecay(
#         initial_learning_rate=0.1,
#         decay_steps=545,
#         decay_rate=0.9,
#         staircase=True
#     )

#     # lr_schedule_pcd = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
#     #     boundaries=[2200, 3000, 3500],
#     #     values=[0.1, 0.01, 0.001, 0.0001]
#     # )
    
#     # Use the learning rate schedule in the optimizer
#     model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=lr_schedule_ed))
#     return model


In [30]:
class RecommendationHyperModel(HyperModel):
    def __init__(self, unique_user_ids, unique_titles, num_genres, rating_weight=1.0, retrieval_weight=1.0):
        self.unique_user_ids = unique_user_ids
        self.unique_titles = unique_titles
        self.num_genres = num_genres
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def build(self, hp):
        embedding_dimension = hp.Int('embedding_dimension', min_value=32, max_value=256, step=32)
        user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
        movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
        genre_input = tf.keras.layers.Input(shape=(self.num_genres,), dtype=tf.float32, name='genres')

        user_lookup = tf.keras.layers.IntegerLookup(vocabulary=self.unique_user_ids, mask_token=None)
        movie_lookup = tf.keras.layers.StringLookup(vocabulary=self.unique_titles, mask_token=None)

        user_embedding = tf.keras.layers.Embedding(len(self.unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
        movie_embedding = tf.keras.layers.Embedding(len(self.unique_titles) + 1, embedding_dimension)(movie_lookup(movie_input))
        genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

        concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

        dense_1 = tf.keras.layers.Dense(hp.Int('units_1', min_value=128, max_value=512, step=64), activation="relu")(concatenated_embeddings)
        dropout_1 = tf.keras.layers.Dropout(hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1))(dense_1)
        dense_2 = tf.keras.layers.Dense(hp.Int('units_2', min_value=64, max_value=256, step=32), activation="relu")(dropout_1)
        dropout_2 = tf.keras.layers.Dropout(hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1))(dense_2)
        rating_output = tf.keras.layers.Dense(1)(dropout_2)

        user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
        movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
        genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
        rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

        metrics = tfrs.metrics.FactorizedTopK(
            candidates=tf.data.Dataset.from_tensor_slices(self.unique_titles).batch(128).map(movie_model)
        )
        rating_task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()],
        )
        retrieval_task = tfrs.tasks.Retrieval(
            metrics=metrics
        )
        model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, self.rating_weight, self.retrieval_weight)
        
        # Define the learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log'),
            decay_steps=hp.Int('decay_steps', min_value=500, max_value=1000, step=100),
            decay_rate=hp.Float('decay_rate', min_value=0.8, max_value=0.9, step=0.05),
            staircase=True
        )

        # # Define the learning rate schedule using CosineDecay
        # lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        #     initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-1, sampling='log'),
        #     decay_steps=1000,
        #     alpha=0.1
        # )
        
        # Use the learning rate schedule in the optimizer
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule))
        return model

In [31]:
# Recreate the tuner
tuner = Hyperband(
    RecommendationHyperModel(unique_user_ids, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_5_categorical_accuracy", direction="max"),
    max_epochs=12,
    factor=3,
    directory='tuning/tpe',
    project_name='20250122063906/movie_recommendation',
)

# Reload the tuner to load previous results
# tuner.reload()

tuner.search(trainds, epochs=12, validation_data=testds, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)])

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
tuned_model = tuner.hypermodel.build(best_hps)

Reloading Tuner from tuning/tpe/20250122063906/movie_recommendation/tuner0.json


In [57]:
# Train the model
history = tuned_model.fit(
    trainds,
    epochs=12,
    # steps_per_epoch=5,
    # validation_data=testds,
    # callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

Epoch 1/12


2025-01-26 21:34:07.051362: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 54603 of 200000
2025-01-26 21:34:17.050726: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 109077 of 200000
2025-01-26 21:34:27.050696: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 164088 of 200000


   1/1091 [..............................] - ETA: 11:45:37 - root_mean_squared_error: 3.4034 - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0078 - factorized_top_k/top_100_categorical_accuracy: 0.0156 - loss: 632.5535 - regularization_loss: 0.0000e+00 - total_loss: 632.5535

2025-01-26 21:34:28.929772: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


1091/1091 [==============================] - 174s 124ms/step - root_mean_squared_error: 0.9336 - factorized_top_k/top_1_categorical_accuracy: 2.5073e-04 - factorized_top_k/top_5_categorical_accuracy: 0.0246 - factorized_top_k/top_10_categorical_accuracy: 0.0553 - factorized_top_k/top_50_categorical_accuracy: 0.1792 - factorized_top_k/top_100_categorical_accuracy: 0.2394 - loss: 605.0421 - regularization_loss: 0.0000e+00 - total_loss: 605.0421
Epoch 2/12
1091/1091 [==============================] - 134s 123ms/step - root_mean_squared_error: 0.8300 - factorized_top_k/top_1_categorical_accuracy: 0.0016 - factorized_top_k/top_5_categorical_accuracy: 0.1173 - factorized_top_k/top_10_categorical_accuracy: 0.1924 - factorized_top_k/top_50_categorical_accuracy: 0.4227 - factorized_top_k/top_100_categorical_accuracy: 0.5299 - loss: 529.4424 - regularization_loss: 0.0000e+00 - total_loss: 529.4424
Epoch 3/12
1091/1091 [==============================] - 145s 133ms/step - root_mean_squared_error: 

In [32]:
best_hps.values

{'embedding_dimension': 224,
 'units_1': 320,
 'dropout_1': 0.4,
 'units_2': 224,
 'dropout_2': 0.4,
 'learning_rate': 0.0022820322663096213,
 'decay_steps': 1000,
 'decay_rate': 0.8500000000000001,
 'tuner/epochs': 12,
 'tuner/initial_epoch': 4,
 'tuner/bracket': 2,
 'tuner/round': 2,
 'tuner/trial_id': '0013'}

In [33]:
def generate_unique_genres(n):
    """
    Generate a one-hot encoded array for `n` unique genres.
    
    Args:
        n (int): Number of unique genres.
    
    Returns:
        np.ndarray: A one-hot encoded array of shape (n, n).
    """
    return np.eye(n, dtype=int).tolist()
    
unique_genres_array = generate_unique_genres(len(unique_genres))

# Extract user and movie embeddings
user_embeddings = tuned_model.user_model.predict(unique_user_ids)
movie_embeddings = tuned_model.movie_model.predict(unique_titles)
genre_embeddings = tuned_model.genre_model.predict(unique_genres_array)


# Create a dictionary to map user IDs and movie IDs to their embeddings
user_embedding_dict = {user_id: embedding for user_id, embedding in zip(unique_user_ids, user_embeddings)}
movie_embedding_dict = {movie_id: embedding for movie_id, embedding in zip(unique_titles, movie_embeddings)}
genre_embedding_dict = {i: embedding for i, embedding in enumerate(genre_embeddings)}

# user_embedding_dict = {k: v for k, v in user_embedding_dict.items()}
movie_embedding_dict = {k.decode('utf-8'): v for k, v in movie_embedding_dict.items()}
# genre_embedding_dict = {k: v for k, v in genre_embedding_dict.items()}

1/1 [==============================] - 0s 147ms/step


In [34]:
# print true if Atlas in movie_embedding_dict
print("Atlas" in movie_embedding_dict) if "Atlas" in movie_embedding_dict else print("Atlas not in movie_embedding_dict")

True


In [35]:
# Convert combined dataset to pandas DataFrame for xgb
xgb_df = pd.DataFrame(combined_dataset.as_numpy_iterator())
print(xgb_df.head())

                          title  user_id  \
0                 b'Initiation'    56703   
1              b'Anything Goes'   125172   
2  b'1000 Miles From Christmas'   129858   
3                   b'Eternals'   174766   
4                   b'Eternals'    55681   

                                              genres  rating  
0  [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...     2.0  
1  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     2.0  
2  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     2.0  
3  [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     2.0  
4  [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     2.0  


In [36]:
# Create feature vectors by combining user, movie, and genres embeddings
def create_feature_vector(row):
    user_id = row['user_id']
    title = row['title'].decode('utf-8')
    genres = row['genres']
    
    if user_id not in user_embedding_dict:
        raise KeyError(f"User ID {user_id} not found in user_embedding_dict")
    if title not in movie_embedding_dict:
        raise KeyError(f"Title '{title}' not found in movie_embedding_dict")
    
    user_embedding = user_embedding_dict[user_id]
    movie_embedding = movie_embedding_dict[title]
    
    return np.concatenate([user_embedding, movie_embedding, genres])

# Apply the function to create feature vectors
try:
    xgb_df['features'] = xgb_df.apply(create_feature_vector, axis=1)
except KeyError as e:
    print(e)
    

# Set the seed for reproducibility
random_seed = 42

# Shuffle the DataFrame
shuffled_df = xgb_df.sample(frac=1, random_state=random_seed).reset_index(drop=True)
train_df, val_df = train_test_split(shuffled_df, test_size=0.2, random_state=random_seed)

# Identify overlapping rows
common_rows = train_df[['user_id', 'title']].merge(val_df[['user_id', 'title']], how='inner')

# Remove overlapping rows from the validation set
if not common_rows.empty:
    val_df = val_df[~val_df[['user_id', 'title']].apply(tuple, axis=1).isin(common_rows.apply(tuple, axis=1))]
    print(str(len(common_rows)) + " overlapping rows removed from the validation set.")

assert train_df[['user_id', 'title']].merge(val_df[['user_id', 'title']], how='inner').empty, "Data leakage detected!"

X_train = np.vstack(train_df['features'].values)
y_train = train_df['rating'].values
X_val = np.vstack(val_df['features'].values)
y_val = val_df['rating'].values

# Add content-based features to the feature matrix
additional_train_features = train_df.drop(columns=['user_id', 'title', 'rating', 'genres', 'features']).values
additional_val_features = val_df.drop(columns=['user_id', 'title', 'rating', 'genres', 'features']).values

# Concatenate the additional features
X_train = np.hstack([X_train, additional_train_features])
X_val = np.hstack([X_val, additional_val_features])

# Verify the shapes of the feature matrices
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")

# Group sizes for ranking
group_train = train_df.groupby('user_id').size().to_list()
group_val = val_df.groupby('user_id').size().to_list()

# Verify that the sum of group sizes matches the number of rows
assert sum(group_train) == X_train.shape[0], "Mismatch between group sizes and number of rows in X_train"
assert sum(group_val) == X_val.shape[0], "Mismatch between group sizes and number of rows in X_val"

# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(group_train)
dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(group_val)

10 overlapping rows removed from the validation set.
Shape of X_train: (139592, 467)
Shape of X_val: (34888, 467)


In [ ]:
print(xgb_df[['title', 'features']])

In [37]:
print(train_df.head())

                                         title  user_id  \
110516                         b'Looop Lapeta'    55707   
149939                            b'Danny Boy'   197432   
96558                   b'Father of the Bride'   156823   
48034            b'The Greatest Beer Run Ever'   173111   
32060   b'Spider-Man: Across the Spider-Verse'   167721   

                                                   genres  rating  \
110516  [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...     1.5   
149939  [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     3.0   
96558   [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     0.5   
48034   [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     5.0   
32060   [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...     5.0   

                                                 features  
110516  [0.0493934266269207, -0.003525983542203903, 0....  
149939  [0.04615713283419609, -0.03819670528173447, -0...  
96558   [0.04274194315075874, 0.010324310511350632

In [38]:
print(val_df.head())

                                          title  user_id  \
70066  b"Are You There God? It's Me, Margaret."   163298   
38489                                   b'Nope'   178415   
7191                                    b'Dune'   105780   
37294                              b'Amsterdam'   142467   
88685                b'Ghostbusters: Afterlife'    68997   

                                                  genres  rating  \
70066  [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     4.0   
38489  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...     2.5   
7191   [1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     3.5   
37294  [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, ...     2.0   
88685  [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     3.5   

                                                features  
70066  [-0.0205727219581604, -0.04557675123214722, -0...  
38489  [-0.008071839809417725, -0.03146757185459137, ...  
7191   [0.0006667599081993103, 0.018841829150915146, 

In [ ]:
# Define the objective function for Optuna
def objective(trial):
    param = {
        'objective': 'rank:pairwise',
        'eval_metric': 'ndcg',
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'lambda': trial.suggest_float('lambda', 0.0, 1.0),
    }
    
    bst = xgb.train(param, dtrain, num_boost_round=100, evals=[(dval, 'eval')], early_stopping_rounds=10, verbose_eval=True)
    y_pred = bst.predict(dval)
    min_rating = 0.5
    max_rating = 5.0
    min_pred = np.min(y_pred)
    max_pred = np.max(y_pred)
    y_pred_scaled = min_rating + (y_pred - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)
    
    # Calculate NDCG score
    ndcg = ndcg_score([y_val], [y_pred_scaled])
    return ndcg

# Create a study and optimize the objective function
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Get the best parameters
best_params = study.best_params

print(f"Best parameters: {best_params}")

# Train the model with the best parameters
bst = xgb.train(best_params, dtrain, num_boost_round=100, evals=[(dval, 'eval')], early_stopping_rounds=10)

In [60]:
best_params

{'eta': 0.22261590340732385,
 'max_depth': 9,
 'min_child_weight': 2,
 'subsample': 0.9438294831775551,
 'colsample_bytree': 0.8199784044119333,
 'gamma': 0.2808066805057597,
 'lambda': 0.9192259491757107}

In [40]:
best_hps.values

{'embedding_dimension': 224,
 'units_1': 320,
 'dropout_1': 0.4,
 'units_2': 224,
 'dropout_2': 0.4,
 'learning_rate': 0.0022820322663096213,
 'decay_steps': 1000,
 'decay_rate': 0.8500000000000001,
 'tuner/epochs': 12,
 'tuner/initial_epoch': 4,
 'tuner/bracket': 2,
 'tuner/round': 2,
 'tuner/trial_id': '0013'}

In [65]:
# Save best_params for xgboost and best_hps for keras tuner
np.save('best_params_xgb.npy', best_params)
np.save('best_hps.npy', best_hps.values)

In [ ]:
# print lenght unique val df titles
print(len(np.unique(list(val_df['title']))))

In [41]:
# Get titles a user already rated
def get_user_titles_rated(user_id):
    try:
        return ratings_bq[ratings_bq['user_id'] == user_id]['title'].values
    except Exception as e:
        print(e)

In [42]:
movies_df = pd.DataFrame(movies.as_numpy_iterator())

In [43]:
def create_rank_feature_vector(user_id, title, genres):
    if user_id not in user_embedding_dict:
        raise KeyError(f"User ID {user_id} not found in user_embedding_dict")
    user_embedding = user_embedding_dict[user_id]

    title = title.decode('utf-8')
    if title not in movie_embedding_dict:
        #size of movie embedding is 224
        movie_embedding = np.zeros(224)
    else:
        movie_embedding = movie_embedding_dict[title]
    
    # Combine user embedding, movie embedding, and genres
    return np.concatenate([user_embedding, movie_embedding, genres])

In [61]:
def rank_movies_with_xgb(user_id, xgb_combined_dataset, bst, k=5, remove_all_rated=True):
    if user_id not in user_embedding_dict:
        raise ValueError(f"User ID {user_id} not found in user_embedding_dict")
    already_rated = get_user_titles_rated(user_id)
    print(f"User {user_id} has rated {len(already_rated)} movies.")
    
    candidate_features = []
    for _, row in xgb_combined_dataset.iterrows():
        title = row['title']
        genres = row['genres']
        additional_features = row.drop(labels=['title', 'genres']).values
        feature_vector = create_rank_feature_vector(user_id, title, genres)
        candidate_features.append(np.concatenate([feature_vector, additional_features]))
    
    candidate_features = np.vstack(candidate_features)
    
    dtest = xgb.DMatrix(candidate_features)
    predicted_ratings = bst.predict(dtest)

    # Scale ratings (optional)
    min_rating, max_rating = 0.5, 5.0
    min_pred = np.min(predicted_ratings)
    max_pred = np.max(predicted_ratings)
    predicted_ratings_scaled = min_rating + (predicted_ratings - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)
    
    ranked_movies = list(zip(predicted_ratings_scaled, xgb_combined_dataset['title'].values))
    print("Ranked Movies Before Filtering:", ranked_movies[:10])  # Print first 10 for debugging
    
    if remove_all_rated:
        # Remove movies that the user has already rated
        ranked_movies = [movie for movie in ranked_movies if movie[1].decode('utf-8') not in already_rated]
        ranked_movies.sort(reverse=True, key=lambda x: x[0])
        return ranked_movies[:k]
    else:
        # Shuffle ranked movies
        random.shuffle(ranked_movies)
        
        # Remove 7 out of every 10 movies that the user has already rated
        filtered_movies = []
        already_rated_count = 0
        for movie in ranked_movies:
            if movie[1].decode('utf-8') in already_rated:
                already_rated_count += 1
                if already_rated_count % 10 < 7:
                    continue  # Skip this movie
            filtered_movies.append(movie)
        
        # Sort the filtered movies by predicted rating in descending order
        filtered_movies.sort(reverse=True, key=lambda x: x[0])
        
        return filtered_movies[:k]


In [62]:
# Example usage
user_id = 163298


top_k_movies = rank_movies_with_xgb(user_id, movies_df, bst, k=5, remove_all_rated=True)
print(f'Ranked movies for user {user_id}: {top_k_movies}')

User 163298 has rated 241 movies.
Ranked Movies Before Filtering: [(3.4702132, b'Siberian Sniper'), (3.1035078, b'Wrong Place Wrong Time'), (3.2274897, b'No Witnesses'), (3.3167372, b'Last of the Wolves'), (2.84263, b'Flinch'), (3.018405, b'The Bezonians'), (3.6925855, b"Everybody's Talking About Jamie"), (4.1562138, b'King Richard'), (3.1230483, b'The King of Laughter'), (3.3242638, b'Granada Nights')]
Ranked movies for user 163298: [(5.0, b'Oppenheimer'), (4.9970155, b'Bo Burnham: Inside'), (4.906022, b'Mountain Flesh'), (4.8524604, b'Puss in Boots: The Last Wish'), (4.820416, b'Coded: The Hidden Love of J.C. Leyendecker')]


In [53]:
# Predict ratings for the validation set
y_pred = bst.predict(dval)
min_rating = 0.5
max_rating = 5.0
min_pred = np.min(y_pred)
max_pred = np.max(y_pred)
y_pred_scaled = min_rating + (y_pred - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)

# # Assuming val_df contains the validation data with columns: user_id, title, rating
# val_df['predicted_rating'] = y_pred_scaled

# # Rank items for each user
# val_df['rank'] = val_df.groupby('user_id')['predicted_rating'].rank(ascending=False, method='first')
# print(pd.Series(y_pred_scaled).describe())

# Calculate NDCG score
ndcg = ndcg_score([y_val], [y_pred_scaled])
print(f'NDCG Score: {ndcg}')

# Determine relevant items with a higher threshold (e.g., rating > 3.5)
def get_relevant_items(user_data, threshold=3.5):
    if user_data['rating'].max() > threshold:
        return user_data[user_data['rating'] > threshold]['title'].values
    else:
        return user_data['title'].values

# Calculate Precision@K and Recall@K and Accuracy@K
def precision_at_k(val_df, k, threshold=3.5):
    precision_scores = []
    for user_id in val_df['user_id'].unique():
        user_data = val_df[val_df['user_id'] == user_id]
        true_titles = get_relevant_items(user_data, threshold)  # Determine relevant items with the new threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        
        # Calculate precision
        precision = len(set(true_titles).intersection(top_k_titles)) / k
        precision_scores.append(precision)
    return np.mean(precision_scores)

def recall_at_k(val_df, k, threshold=3.5):
    recall_scores = []
    for user_id in val_df['user_id'].unique():
        user_data = val_df[val_df['user_id'] == user_id]
        true_titles = get_relevant_items(user_data, threshold)  # Determine relevant items with the new threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        
        # Calculate recall
        recall = len(set(true_titles).intersection(top_k_titles)) / len(true_titles)
        recall_scores.append(recall)
    return np.mean(recall_scores)

# Calculate Accuracy@K
def accuracy_at_k(val_df, k, threshold=3.5):
    correct_count = 0
    total_users = val_df['user_id'].nunique()
    
    for user_id in val_df['user_id'].unique():
        user_data = val_df[val_df['user_id'] == user_id]
        true_titles = get_relevant_items(user_data, threshold)  # Determine relevant items with the new threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        
        # Check if any true title is in the top-K predictions
        if set(true_titles).intersection(top_k_titles):
            correct_count += 1
    
    return correct_count / total_users

# Evaluate the impact of changing the threshold
threshold = 2.5
precision_10 = precision_at_k(val_df, 10, threshold)
recall_10 = recall_at_k(val_df, 10, threshold)
accuracy_10 = accuracy_at_k(val_df, 10, threshold)

print(f'Precision@10 with threshold {threshold}: {precision_10}')
print(f'Recall@10 with threshold {threshold}: {recall_10}')
print(f'Accuracy@10 with threshold {threshold}: {accuracy_10}')

NDCG Score: 0.98810065438214
Precision@10 with threshold 2.5: 0.2816014449127032
Recall@10 with threshold 2.5: 0.9720524338328527
Accuracy@10 with threshold 2.5: 0.9998795906080674


In [ ]:

# Additional diagnostics
print("Distribution of Ratings in Training Set:")
print(train_df['rating'].value_counts())

print("Distribution of Ratings in Validation Set:")
print(val_df['rating'].value_counts())

print("Predictions Distribution:")
print(pd.Series(y_pred_scaled).describe())

In [54]:
# print genre for given movieId using movies_bq
def get_genre(title):
    try:
        return movies_bq[movies_bq['title'] == title]['genres'].values[0]
    except Exception as e:
        print(e)
        
def rate_movies_with_hypermodel(hypermodel, user_id, titles, genres):
    # Use the recommendation hypermodel to predict ratings for the given user and movie titles
    predicted_ratings = []
    try:
        for title, genre in zip(titles, genres):
            # Predict ratings using the hypermodel
            _, _, _, predicted_rating = hypermodel({
                "user_id": np.array([user_id]),
                "title": np.array([title]),
                "genres": np.array([genre])
            })
            predicted_ratings.append([title, predicted_rating.numpy()[0][0]])
        return predicted_ratings
    except Exception as e:
        print(e)

movie_titles = ['The Rescue', 'Siberian Sniper']
movie_genres = [get_genre(title) for title in movie_titles]
predicted_ratings = rate_movies_with_hypermodel(tuned_model, user_id, movie_titles, movie_genres)
print("Predicted ratings:")
print(predicted_ratings)

Predicted ratings:
[['The Rescue', 0.06805917], ['Siberian Sniper', 0.015595373]]


In [63]:
def get_final_predictions(user_id, combined_dataset, bst, hypermodel, k=5, remove_all_rated=True):
    # Rank movies using the XGBoost model
    num_retrieval = k * 100 if k < 6 else 500
    top_k_movies = rank_movies_with_xgb(user_id, combined_dataset, bst, k=num_retrieval, remove_all_rated=remove_all_rated)

    # Get the movie titles and genres
    _, movie_titles = zip(*top_k_movies)

    # Decode the movie titles
    movie_titles = [title.decode('utf-8') for title in movie_titles]
    movie_genres = [get_genre(title) for title in movie_titles]

    # Predict ratings using the hypermodel
    final_predictions = rate_movies_with_hypermodel(hypermodel, user_id, movie_titles, movie_genres)

    # Sort the final predictions by rating
    final_predictions = sorted(final_predictions, key=lambda x: x[1], reverse=True)

    return final_predictions[:k]

In [64]:
final_predictions = get_final_predictions(user_id, movies_df, bst, tuned_model, k=3, remove_all_rated=True)
print("Final Predictions:")
print(final_predictions)

User 163298 has rated 241 movies.
Ranked Movies Before Filtering: [(3.4702132, b'Siberian Sniper'), (3.1035078, b'Wrong Place Wrong Time'), (3.2274897, b'No Witnesses'), (3.3167372, b'Last of the Wolves'), (2.84263, b'Flinch'), (3.018405, b'The Bezonians'), (3.6925855, b"Everybody's Talking About Jamie"), (4.1562138, b'King Richard'), (3.1230483, b'The King of Laughter'), (3.3242638, b'Granada Nights')]
Final Predictions:
[['Who We Are: A Chronicle of Racism in America', 4.3162007], ['In Search of Tomorrow', 3.8172963], ['Chernobyl: The Lost Tapes', 3.7620506]]


In [48]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/tpe/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_5_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(f"trained_model/tpe/tensorboard/{timestamp}_cp.ckpt", histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

In [31]:
model = create_model(unique_user_ids, unique_titles, len(unique_genres), rating_weight=1.0, retrieval_weight=1.0)

In [ ]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/bayesian/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_5_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(f"trained_model/bayesian/tensorboard/{timestamp}_cp.ckpt", histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

bayesian_tuner = BayesianOptimization(
    RecommendationHyperModel(unique_user_ids, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_5_categorical_accuracy", direction="max"),
    max_trials=10,
    executions_per_trial=1,
    directory='tuning/bayesian_optimization',
    project_name='20250120133706/movie_recommendation',
)
bayesian_tuner.reload()
# Get the best hyperparameters
best_hps = bayesian_tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
bayesian_model = bayesian_tuner.hypermodel.build(best_hps)
# Build the model by calling it on a batch of data
bayesian_model.fit(
    trainds,
    epochs=10,
    # validation_data=testds,
    callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

In [ ]:
# Build the model by calling it on a batch of data
model.build(input_shape={
	"user_id": (None,),
	"title": (None,),
	"genres": (None, len(unique_genres))
})

In [ ]:
# Train the model
history = tuned_model.fit(
    trainds,
    epochs=12,
    # steps_per_epoch=5,
    # validation_data=testds,
    # callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

In [59]:
steps = ds_length * 0.2 // test_batch_size
metrics = tuned_model.evaluate(testds, steps=steps, return_dict=True)

2025-01-26 22:15:39.116075: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 31283 of 200000
2025-01-26 22:15:49.116129: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 63901 of 200000
2025-01-26 22:15:59.116278: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 91713 of 200000
2025-01-26 22:16:09.116229: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 119929 of 200000
2025-01-26 22:16:19.116087: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 142735 of 200000
2025-01-26 22:16:29.116113: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 170276 of 200000
2025-01-26 22:16:30.809252: I tensorflow/core/kernels/data/shuffle_dataset_op.c

545/545 [==============================] - 123s 109ms/step - root_mean_squared_error: 0.6768 - factorized_top_k/top_1_categorical_accuracy: 0.0421 - factorized_top_k/top_5_categorical_accuracy: 0.2333 - factorized_top_k/top_10_categorical_accuracy: 0.3589 - factorized_top_k/top_50_categorical_accuracy: 0.6298 - factorized_top_k/top_100_categorical_accuracy: 0.7221 - loss: 163.3529 - regularization_loss: 0.0000e+00 - total_loss: 163.3529


In [ ]:
print(f"Retrieval top-1 accuracy: {metrics['factorized_top_k/top_1_categorical_accuracy']:.3f}.")
print(f"Retrieval top-5 accuracy: {metrics['factorized_top_k/top_5_categorical_accuracy']:.3f}.")
print(f"Retrieval top-10 accuracy: {metrics['factorized_top_k/top_10_categorical_accuracy']:.3f}.")
print(f"Ranking RMSE: {metrics['root_mean_squared_error']:.3f}.")

In [66]:
tuned_model.save_weights(f"trained_model/tpe/weights/20250122063906_weights.h5")

In [68]:
# Save xgb model
bst.save_model(f"trained_model/xgb/20250122063906_xgb_model.json")

In [ ]:


trained_user_embeddings, trained_movie_embeddings, trained_genre_embeddings, predicted_rating = tuned_model({
      "user_id": np.array([163298]),
      "title": np.array(['The Banshees of Inisherin']),
      "genres": np.array([get_genre('The Banshees of Inisherin')])
  })
print("Predicted rating:")
print(predicted_rating)

In [174]:
def extract_embeddings(model, unique_user_ids, unique_titles):
    """
    Extract embeddings for all users and movies using the trained model.
    
    Args:
    - model: The trained recommendation model.
    - unique_user_ids: List of unique user IDs.
    - unique_movie_ids: List of unique movie IDs.
    
    Returns:
    - user_embeddings: Embeddings for all users.
    - movie_embeddings: Embeddings for all movies.
    """
    # Extract movie embeddings
    titles = np.array(unique_titles)
    movie_embeddings = model.movie_model(tf.constant(titles, dtype=tf.string)).numpy()

    # Extract user embeddings
    user_ids = np.array(unique_user_ids)
    user_embeddings = model.user_model(tf.constant(user_ids, dtype=tf.int32)).numpy()

    return user_embeddings, movie_embeddings

In [175]:
def index_movie_embeddings(movie_embeddings):
    """
    Index the movie embeddings using FAISS.
    
    Args:
    - movie_embeddings: Embeddings for all movies.
    
    Returns:
    - index: FAISS index with movie embeddings.
    """
    # Dimension of the embeddings
    embedding_dimension = movie_embeddings.shape[1]

    # Create a FAISS index
    index = faiss.IndexFlatL2(embedding_dimension)

    # Add movie embeddings to the index
    index.add(movie_embeddings)

    return index

In [176]:
def recommend_movies(model, index, unique_titles, user_id, k=10):
    """
    Recommend movies for a given user by querying the FAISS index.
    
    Args:
    - model: The trained recommendation model.
    - index: FAISS index with movie embeddings.
    - unique_titles: List of unique movie titles.
    - user_id: The user ID for which to make recommendations.
    - k: Number of recommendations to retrieve (default is 10).
    
    Returns:
    - recommended_movie_ids: List of recommended movie IDs.
    """
    # Get the embedding for the given user
    user_embedding = model.user_model(tf.constant([user_id], dtype=tf.int32)).numpy()

    # Query the FAISS index
    distances, indices = index.search(user_embedding, k)

    # Get the recommended movie IDs
    recommended_movies = np.array(unique_titles)[indices[0]]

    return recommended_movies

In [ ]:
# Extract embeddings
user_embeddings, movie_embeddings = extract_embeddings(tuned_model, unique_user_ids, unique_titles)

# Index movie embeddings
index = index_movie_embeddings(movie_embeddings)

# Example user ID for which to make recommendations
example_user_id = 163298

recommended_movie_ids = recommend_movies(tuned_model, index, unique_titles, example_user_id, k=5)

print(f"Recommended movie titles for user {example_user_id}: {recommended_movie_ids}")

In [61]:
faiss.write_index(index, "trained_model/faiss/{}_faiss.index".format(timestamp))